# GENERATE NEW NAMES USING AI. PARTICULARLY MLP
***

Preparing data

In [1]:
with open("data/names.txt","r") as file:
    data = file.read().splitlines()

In [2]:
alphabets = list(set("".join(data)))
alphabets.sort()

stoi = {k:v for v,k in enumerate(alphabets)}
stoi["."] = 26

itos = {v:k for k,v in stoi.items()}

In [3]:
#Iterate through words

for word in data[:1]:
    chars = "..."+ word + "."

    for char1,char2,char3,char4 in zip(chars,chars[1:],chars[2:],chars[3:]):
        print(char1,char2,char3,"->",char4)

. . . -> e
. . e -> m
. e m -> m
e m m -> a
m m a -> .


# Creating MLP dataset

In [62]:
import torch
import torch.nn.functional as F

In [91]:
Xs = []
ys = []
for word in data:
    chars = "..."+ word + "."

    for char1,char2,char3,char4 in zip(chars,chars[1:],chars[2:],chars[3:]):
        idx1 = stoi[char1]
        idx2 = stoi[char2]
        idx3 = stoi[char3]
        idx4 = stoi[char4]

        Xs.append([idx1,idx2,idx3])
        ys.append(idx4)

Xs = torch.tensor(Xs)
ys = torch.tensor(ys)

vocab_size = len(stoi)

In [92]:
X_enc = F.one_hot(Xs,num_classes=vocab_size).to(torch.float)
Y_enc = F.one_hot(ys,num_classes=vocab_size).to(torch.float)

In [93]:
Y_enc.shape

torch.Size([228146, 27])

In [94]:
X_enc.dtype

torch.float32

In [95]:
feature_space = 3
batch_size = 10
h_in = feature_space * 3
h_out = 100

Emb = torch.randn((vocab_size,feature_space))
W1 = torch.randn((h_in,h_out))
b1 = torch.randn((1,h_out))

W2 = torch.randn((h_out,vocab_size)) * 0.01
b2 = torch.randn((1,vocab_size)) * 0.0001


parameters = [Emb,W1,b1,W2,b2]

for p in parameters:
    p.requires_grad = True

batch_idx = torch.randint(0,Xs.shape[0],(batch_size,))

X_enc[batch_idx].shape

torch.Size([10, 3, 27])

In [96]:
print(f"Size of model : {sum(p.numel() for p in parameters)}")

Size of model : 3808


In [97]:
# forward pass

batch_idx = torch.randint(0,Xs.shape[0],(batch_size,))
X = X_enc[batch_idx]
y = Y_enc[batch_idx]
X_emb = X @ Emb
h_preact = X_emb.view((-1,h_in)) @ W1  + b1
h = torch.tanh(h_preact)

o = h @ W2 + b2
loss = F.cross_entropy(o,y)

In [98]:
loss.item()

3.277439832687378

In [101]:
#Full Training

epochs = 1000
log_every = 50
alpha = 0.1
batch_size = 512

for epoch in range(epochs):
    batch_idx = torch.randint(0,Xs.shape[0],(batch_size,))
    X = X_enc[batch_idx]
    y = Y_enc[batch_idx]
    X_emb = X @ Emb
    h_preact = X_emb.view((-1,h_in)) @ W1  + b1
    h = torch.tanh(h_preact)

    o = h @ W2 + b2
    loss = F.cross_entropy(o,y)

    # Zero grad

    for p in parameters:
        p.grad = None

    loss.backward()

    # Update Weights

    for p in parameters:
        p.data -= alpha * p.grad

    if epoch % log_every == 0:
        print(f"Epoch : {epoch} Loss : {loss.item()} ")

    


Epoch : 0 Loss : 2.5051283836364746 
Epoch : 50 Loss : 2.5018460750579834 
Epoch : 100 Loss : 2.4101996421813965 
Epoch : 150 Loss : 2.448961019515991 
Epoch : 200 Loss : 2.4979450702667236 
Epoch : 250 Loss : 2.413677453994751 
Epoch : 300 Loss : 2.378356456756592 
Epoch : 350 Loss : 2.4858524799346924 
Epoch : 400 Loss : 2.411454439163208 
Epoch : 450 Loss : 2.3677263259887695 
Epoch : 500 Loss : 2.3858895301818848 
Epoch : 550 Loss : 2.4459104537963867 
Epoch : 600 Loss : 2.4065847396850586 
Epoch : 650 Loss : 2.42862868309021 
Epoch : 700 Loss : 2.3372535705566406 
Epoch : 750 Loss : 2.485644817352295 
Epoch : 800 Loss : 2.4200024604797363 
Epoch : 850 Loss : 2.4295847415924072 
Epoch : 900 Loss : 2.3393008708953857 
Epoch : 950 Loss : 2.3625147342681885 
